# 🚗 Driver Monitoring System — YOLOv8 Training
**Cairo University | FCAI | AI Department | 2026**

This notebook trains YOLOv8 to detect:
- 🚬 Cigarette / Smoking
- 📱 Phone usage

**Runtime: GPU (T4 recommended) — Training takes ~15-20 min**

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────
!pip install ultralytics roboflow -q

In [ ]:
# ── Step 2: Check GPU ─────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
!nvidia-smi

In [ ]:
# ── Step 3: Download Smoking Detection Dataset from Roboflow ──────────────
# Dataset: Smoking Detection (2,000+ annotated images)
from roboflow import Roboflow

# Using public dataset — no API key needed for public datasets
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Get free key at roboflow.com
project = rf.workspace("roboflow-universe-projects").project("cigarette-detection-xokxa")
version = project.version(2)
dataset = version.download("yolov8")

print(f'Dataset downloaded to: {dataset.location}')

In [ ]:
# ── ALTERNATIVE: Use this if no Roboflow key ──────────────────────────────
# Download pre-prepared dataset directly
import os

# Create dataset structure manually with sample data for demo
os.makedirs('dataset/train/images', exist_ok=True)
os.makedirs('dataset/train/labels', exist_ok=True)
os.makedirs('dataset/val/images',   exist_ok=True)
os.makedirs('dataset/val/labels',   exist_ok=True)

# Create data.yaml
yaml_content = """
path: /content/dataset
train: train/images
val:   val/images

nc: 2
names: ['cigarette', 'phone']
"""

with open('dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print('Dataset structure created!')

In [ ]:
# ── Step 4: Train YOLOv8 ─────────────────────────────────────────────────
from ultralytics import YOLO

# Load pretrained YOLOv8n (nano — fast and lightweight)
model = YOLO('yolov8n.pt')

# Fine-tune on our dataset
results = model.train(
    data    = 'dataset/data.yaml',
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    name    = 'driver_monitor',
    project = 'runs/train',
    device  = 0,          # GPU
    patience = 10,        # early stopping
    save    = True,
    plots   = True
)

print('Training complete!')
print(f'Best weights: {results.save_dir}/weights/best.pt')

In [ ]:
# ── Step 5: Evaluate model ────────────────────────────────────────────────
best_model = YOLO('runs/train/driver_monitor/weights/best.pt')

metrics = best_model.val(data='dataset/data.yaml')
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall   : {metrics.box.mr:.4f}')

In [ ]:
# ── Step 6: Plot training results ─────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd

results_csv = pd.read_csv('runs/train/driver_monitor/results.csv')
results_csv.columns = results_csv.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('YOLOv8 Training Results — Driver Monitoring System', fontsize=14)

axes[0,0].plot(results_csv['train/box_loss']); axes[0,0].set_title('Train Box Loss')
axes[0,1].plot(results_csv['train/cls_loss']); axes[0,1].set_title('Train Class Loss')
axes[0,2].plot(results_csv['metrics/mAP50(B)']); axes[0,2].set_title('mAP@0.5')
axes[1,0].plot(results_csv['val/box_loss']);   axes[1,0].set_title('Val Box Loss')
axes[1,1].plot(results_csv['metrics/precision(B)']); axes[1,1].set_title('Precision')
axes[1,2].plot(results_csv['metrics/recall(B)']); axes[1,2].set_title('Recall')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150)
plt.show()
print('Plot saved!')

In [ ]:
# ── Step 7: Test on a sample image ───────────────────────────────────────
import cv2
from google.colab.patches import cv2_imshow

# Run inference on a test image
test_results = best_model.predict(
    source='dataset/val/images',
    conf=0.45,
    save=True
)

# Show first result
result_img = cv2.imread(str(test_results[0].save_dir / 'image0.jpg'))
if result_img is not None:
    cv2_imshow(result_img)

In [ ]:
# ── Step 8: Download the trained model ───────────────────────────────────
from google.colab import files
import shutil

# Copy best weights
shutil.copy('runs/train/driver_monitor/weights/best.pt', 'cigarette_model.pt')

# Download
files.download('cigarette_model.pt')
files.download('training_results.png')

print('Downloaded! Place cigarette_model.pt in models/weights/')